# 02 — Preprocessing & SMOTE
## Diabetes Risk Stratification · CDC BRFSS 2015

**Key decision in this notebook:** How to handle the 46:1 imbalance between the majority class and prediabetes. Standard SMOTE to full balance with 170k samples per class is computationally infeasible. We use **targeted SMOTE** — oversample only prediabetes to 20k, leave the other classes as-is — and justify this decision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('OK')

## 1. Load & Cast

In [ ]:
df = pd.read_csv('../data/diabetes_012_health_indicators_BRFSS2015.csv')
df['Diabetes_012'] = df['Diabetes_012'].astype(int)

X = df.drop('Diabetes_012', axis=1)
y = df['Diabetes_012']
print(f'X shape: {X.shape}, y distribution: {Counter(y)}')

## 2. Train/Test Split

Stratified split ensures each class is proportionally represented in train and test — critical with severe imbalance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train class dist: {Counter(y_train)}')
print(f'Test class dist:  {Counter(y_test)}')

## 3. Scaling

Fit StandardScaler on training data only, apply to both splits.

In [ ]:
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

joblib.dump(scaler, '../src/scaler.pkl')
print('Scaler saved. Feature means (should be ~0):')  
print(X_train_sc.mean().round(3).head())

## 4. SMOTE — Targeted Oversampling

**Decision rationale:**

Full SMOTE to 170k samples per class would require ~500M+ synthetic samples and exceed available memory. More importantly, generating 165k synthetic prediabetes samples from 3,705 real ones introduces significant synthetic noise — too far from the original distribution.

**Targeted strategy:** oversample prediabetes to 20k (~5.4x) and leave majority classes as-is. This gives the model enough prediabetes signal while staying computationally feasible and closer to real data distribution.

**SMOTE is applied ONLY to training data** to prevent data leakage into the test set.

In [ ]:
sampling_strategy = {
    0: Counter(y_train)[0],   # majority: unchanged
    1: 20000,                  # prediabetes: oversample to 20k
    2: Counter(y_train)[2],   # diabetes: unchanged
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print(f'Before SMOTE: {Counter(y_train)}')
print(f'After SMOTE:  {Counter(y_train_res)}')
print(f'\nSmote multiplier on prediabetes: {20000/Counter(y_train)[1]:.1f}x')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['No diabetes', 'Prediabetes', 'Diabetes']
for ax, (counts, title) in zip(axes, [
    (Counter(y_train), 'Before SMOTE'),
    (Counter(y_train_res), 'After SMOTE')
]):
    ax.bar(labels, [counts[0], counts[1], counts[2]],
           color=['#4C72B0','#FF8C00','#C44E52'], alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_yscale('log')
plt.suptitle('Class Distribution Before and After Targeted SMOTE', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Save Artifacts

In [ ]:
X_train_res.to_csv('../data/X_train_resampled.csv', index=False)
pd.Series(y_train_res, name='Diabetes_012').to_csv('../data/y_train_resampled.csv', index=False)
X_test_sc.to_csv('../data/X_test_scaled.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
pd.Series(y_test, name='Diabetes_012').to_csv('../data/y_test.csv', index=False)
joblib.dump(list(X_train.columns), '../src/feature_names.pkl')

print('All preprocessing artifacts saved.')
print('\n→ Proceed to notebook 03: Model Training & Evaluation')